# 03 — Li, Liu, Liu & Wei (2024), "Replicating and digesting anomalies in the Chinese A-share market"

All 469 anomalies, four portfolio procedures, seven factor models (lighthouse.md 5.3). Logic lives in `src/replication/li/`; this notebook only calls it and shows ours vs the paper. Run `python -m replication.li --build` first to fill the anomaly caches (aggregates only).

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 200)
from replication.li import tables as T, catalog as C
from replication.li.__main__ import TABLE_FUNCS
from replication.common.compare import compare
segs = T.li_segments(); orig = segs['original']

In [2]:
ours = pd.concat([f(orig) for f in TABLE_FUNCS.values()], ignore_index=True)
cmp = compare('li', ours)
scored = cmp[cmp['pass'].notna()]
summary = scored.groupby('table')['pass'].agg(['size', 'sum'])
summary.columns = ['scored', 'within tolerance']
summary

,scored,within tolerance
table,,
T1,530,526
T2,208,143
T3,168,52
T4,280,142
T5,392,151
T6,2224,1348
T7,448,139
TXT,20,9


In [3]:
def show(table, rows=None):
    x = cmp[cmp['table'] == table][['row', 'col', 'paper', 'ours', 'kind', 'pass']].copy()
    if rows: x = x[x['row'].str.match(rows)]
    x['pass'] = x['pass'].map({True: 'ok', False: 'MISS'}).fillna('')
    return x

## Table 1: sample and factors

In [4]:
show('T1', r'A:(1999|2005|2018)')

,row,col,paper,ours,kind,pass
0,A:1999,n_ashares,844.00,844.000000,count,ok
1,A:1999,n_sh_main,434.00,434.000000,count,ok
2,A:1999,n_sz_main,410.00,410.000000,count,ok
3,A:1999,n_sme,0.00,0.000000,count,ok
4,A:1999,n_gem,0.00,0.000000,count,ok
5,A:1999,tmv,2.40,2.404137,rel10,ok
6,A:1999,cmv,0.73,0.728271,rel10,ok
7,A:1999,sh_tmv,1.32,1.320716,rel10,ok
8,A:1999,sh_cmv,0.39,0.385134,rel10,ok
9,A:1999,sz_tmv,1.08,1.083421,rel10,ok


In [5]:
show('T1', r'[BC]:')

,row,col,paper,ours,kind,pass
300,B:MKT,mean,0.62,0.618148,pct_return,ok
301,B:ME,mean,0.78,0.774704,pct_return,ok
302,B:I/A,mean,0.29,0.263555,pct_return,ok
303,B:ROE,mean,0.54,0.489894,pct_return,ok
304,B:SMB-FF5,mean,0.68,0.678354,pct_return,ok
305,B:HML-FF5,mean,0.42,0.324150,pct_return,ok
306,B:RMW,mean,-0.14,-0.263670,pct_return,MISS
307,B:CMA,mean,0.05,0.077460,pct_return,ok
308,B:MKT-CH,mean,0.46,0.456404,pct_return,ok
309,B:SMB-CH3,mean,0.73,0.725356,pct_return,ok


## Table 2: microcaps and investment capacity

In [6]:
show('T2', r'A:')

,row,col,paper,ours,kind,pass
534,A:Market,n_firms,1563.00,1529.771930,count,ok
535,A:Market,pct_tmv,100.00,100.000000,share_pct,ok
536,A:Market,pct_cmv,100.00,100.000000,share_pct,ok
537,A:Market,vw_mean,0.80,0.748222,pct_return,ok
538,A:Market,vw_std,8.63,8.513647,rel10,ok
539,A:Market,ew_mean,1.28,1.253634,pct_return,ok
540,A:Market,ew_std,9.84,9.697844,rel10,ok
541,A:Market,xs_std,9.88,10.762314,rel10,ok
542,A:Micro,n_firms,449.00,459.258772,count,ok
543,A:Micro,pct_tmv,8.78,9.285274,share_pct,ok


In [7]:
show('T2', r'[BCD]:Mainboard-VW')

,row,col,paper,ours,kind,pass
574,B:Mainboard-VW,Low-All,8.20,7.686724,share_pct,ok
575,B:Mainboard-VW,Low-Mom,6.87,6.999688,share_pct,ok
576,B:Mainboard-VW,Low-VvG,6.87,8.771783,share_pct,ok
577,B:Mainboard-VW,Low-Inv,8.40,7.584353,share_pct,ok
578,B:Mainboard-VW,Low-Prof,11.94,9.711756,share_pct,ok
579,B:Mainboard-VW,Low-Intan,7.60,6.402514,share_pct,ok
580,B:Mainboard-VW,Low-Fric,6.15,6.716180,share_pct,ok
581,B:Mainboard-VW,High-All,6.80,6.660975,share_pct,ok
582,B:Mainboard-VW,High-Mom,6.25,4.045897,share_pct,ok
583,B:Mainboard-VW,High-VvG,5.56,8.074795,share_pct,ok


## Table 3: significance rates

In [8]:
show('T3', r'A:')

,row,col,paper,ours,kind,pass
742,A:Mainboard-VW,Momentum:SHT,15.56,28.888889,share_pct,MISS
743,A:Mainboard-VW,Momentum:MHT,4.44,8.888889,share_pct,MISS
744,A:Mainboard-VW,Value-versus-growth:SHT,8.82,2.941176,share_pct,MISS
745,A:Mainboard-VW,Value-versus-growth:MHT,0.00,1.470588,share_pct,ok
746,A:Mainboard-VW,Investment:SHT,2.78,5.555556,share_pct,ok
747,A:Mainboard-VW,Investment:MHT,0.00,2.777778,share_pct,ok
748,A:Mainboard-VW,Profitability:SHT,8.51,24.468085,share_pct,MISS
749,A:Mainboard-VW,Profitability:MHT,5.32,4.255319,share_pct,ok
750,A:Mainboard-VW,Intangibles:SHT,12.05,13.414634,share_pct,ok
751,A:Mainboard-VW,Intangibles:MHT,2.41,3.658537,share_pct,ok


## Table 4: significant anomalies (Mainboard-VW)

In [9]:
show('T4', r'A:count')

,row,col,paper,ours,kind,pass
2314,A:count,significant,78.0,79.0,count,ok


In [10]:
t4 = show('T4', r'A:'); t4[t4['col'] == 'm'].head(80)

,row,col,paper,ours,kind,pass
910,A:sue_q_1,m,1.17,1.118640,pct_return,ok
928,A:sue_q_3,m,0.82,0.704521,pct_return,MISS
946,A:sue_q_6,m,0.54,0.485765,pct_return,ok
964,A:sue_q_9,m,0.47,0.377268,pct_return,ok
982,A:abr_q_9,m,0.32,0.258522,pct_return,ok
1000,A:_6_res_mom_3,m,0.56,0.610472,pct_return,ok
1018,A:_6_res_mom_6,m,0.44,0.483659,pct_return,ok
1036,A:ep_q_1,m,0.84,1.026501,pct_return,MISS
1054,A:ndp,m,0.68,0.308989,pct_return,MISS
1072,A:ndp_q_3,m,0.74,0.383080,pct_return,MISS


## Table 5: factor models

In [11]:
show('T5', r'A:')

,row,col,paper,ours,kind,pass
3414,A:CAPM,All:abs_alpha_hl,0.85,0.703205,pct_return,MISS
3415,A:CAPM,All:n_t196,74.00,78.000000,count,MISS
3416,A:CAPM,All:mean_abs_alpha,0.39,0.265252,pct_return,MISS
3417,A:CAPM,All:n_grs_p05,65.00,49.000000,count,MISS
3418,A:CAPM,Momentum:abs_alpha_hl,0.63,0.585196,pct_return,ok
3419,A:CAPM,Momentum:n_t196,7.00,13.000000,count,MISS
3420,A:CAPM,Momentum:mean_abs_alpha,0.32,0.228039,pct_return,ok
3421,A:CAPM,Momentum:n_grs_p05,4.00,11.000000,count,MISS
3422,A:CAPM,Value-versus-growth:abs_alpha_hl,0.72,1.027594,pct_return,MISS
3423,A:CAPM,Value-versus-growth:n_t196,4.00,2.000000,count,MISS


## Table 6 (excerpt) and Table 7

In [12]:
show('T6', r'A:sue_q_1')

,row,col,paper,ours,kind,pass
912,A:sue_q_1,alpha_FF5,1.11,1.282842,pct_return,MISS
913,A:sue_q_1,t_FF5,4.36,5.275040,t,ok
914,A:sue_q_1,mean_abs_alpha_FF5,0.38,0.336778,pct_return,ok
915,A:sue_q_1,grs_p_FF5,0.00,0.000002,pvalue,ok
916,A:sue_q_1,alpha_q,0.78,0.802633,pct_return,ok
917,A:sue_q_1,t_q,3.10,3.432994,t,ok
918,A:sue_q_1,mean_abs_alpha_q,0.21,0.238910,pct_return,ok
919,A:sue_q_1,grs_p_q,0.07,0.017605,pvalue,MISS
920,A:sue_q_1,alpha_CH3,0.47,0.515233,pct_return,ok
921,A:sue_q_1,t_CH3,1.70,1.948077,t,ok


In [13]:
show('T7', r'[AC]:Total')

,row,col,paper,ours,kind,pass
3902,A:Total,0407:Raw,44.0,43.0,count,ok
3903,A:Total,0407:CAPM,40.0,40.0,count,ok
3904,A:Total,0407:FF3,33.0,35.0,count,MISS
3905,A:Total,0407:FF5,36.0,37.0,count,ok
3906,A:Total,0407:q,24.0,18.0,count,MISS
3907,A:Total,0407:CH3,20.0,14.0,count,MISS
3908,A:Total,0407:CH4,24.0,11.0,count,MISS
3909,A:Total,0407:Car4,39.0,36.0,count,MISS
3910,A:Total,0807:Raw,44.0,49.0,count,MISS
3911,A:Total,0807:CAPM,40.0,46.0,count,MISS


## Text numbers and section 4.2

In [14]:
show('TXT')

,row,col,paper,ours,kind,pass
4254,abstract:failure_rate_raw,pct,83.37,82.560706,share_pct,ok
4255,abstract:failure_rate_capm,pct,84.22,82.781457,share_pct,ok
4256,abstract:failure_rate_ff3,pct,86.99,85.430464,share_pct,ok
4257,S1:sig_full_sht,n,78.00,79.000000,count,ok
4258,S1:sig_full_mht,n,29.00,19.000000,count,MISS
4259,S1:sig_post2007_sht,n,61.00,58.000000,count,MISS
4260,S1:sig_post2007_mht,n,22.00,18.000000,count,MISS
4261,S1:explained_full_CH3,pct,53.85,73.417722,share_pct,MISS
4262,S1:explained_full_CH4,pct,47.44,73.417722,share_pct,MISS
4263,S1:explained_full_q,pct,25.64,54.430380,share_pct,MISS


## Extension (lighthouse 6): same functions, later years

The extended and full segments headline the look-ahead-free version (`portfolios.HOLDING_MONTH_FILTER = False`, user decision 2026-10-11); the authors' procedure is shown next to it.

In [15]:
from csmar.registry import PROJECT_ROOT
R = PROJECT_ROOT / 'results' / 'li'
def headline(seg, variant):
    t = pd.read_csv(R / f"{seg}_tables{'' if variant == 'authors' else '_clean'}.csv")
    g = lambda tb, r, c: t[(t['table'] == tb) & (t['row'] == r) & (t['col'] == c)]['value'].iloc[0]
    return {'segment': seg, 'variant': variant,
            'share significant SHT (%)': round(g('T3', 'A:Mainboard-VW', 'All categories:SHT'), 2),
            'share significant MHT (%)': round(g('T3', 'A:Mainboard-VW', 'All categories:MHT'), 2),
            'n significant': int(g('T4', 'A:count', 'significant')),
            'VMG mean (%)': round(g('T1', 'B:VMG', 'mean'), 2) if len(t[(t['table']=='T1')&(t['row']=='B:VMG')]) else None}
# Headline per lighthouse 6 and the user's decision: authors' procedure for the original segment,
# look-ahead-free ("clean") for the extended and full segments; the other variant alongside.
pd.DataFrame([headline(s, v) for s in ('original', 'extended', 'full') for v in (('authors', 'clean') if s == 'original' else ('clean', 'authors'))])

,segment,variant,share significant SHT (%),share significant MHT (%),n significant,VMG mean (%)
0,original,authors,17.44,4.19,79,1.20
1,original,clean,26.71,9.27,121,1.20
2,extended,clean,4.42,1.10,20,0.46
3,extended,authors,2.65,1.10,12,0.46
4,full,clean,24.06,8.17,109,1.12
5,full,authors,14.57,4.42,66,1.12


### Extended and full segments (look-ahead-free headline): significance by category and factor-model summary

In [16]:
for seg in ('extended', 'full'):
    t = pd.read_csv(R / f'{seg}_tables_clean.csv')
    x = t[(t['table'] == 'T3') & (t['row'] == 'A:Mainboard-VW')].set_index('col')['value'].round(2)
    print(seg); display(x.to_frame().T)
    t5 = t[(t['table'] == 'T5') & t['row'].str.startswith('A:') & t['col'].str.startswith('All:')]
    display(t5.pivot(index='row', columns='col', values='value').round(2))

extended


col,Momentum:SHT,Momentum:MHT,Value-versus-growth:SHT,Value-versus-growth:MHT,Investment:SHT,Investment:MHT,Profitability:SHT,Profitability:MHT,Intangibles:SHT,Intangibles:MHT,Trading frictions:SHT,Trading frictions:MHT,All categories:SHT,All categories:MHT
value,8.89,0.0,0.0,0.0,5.56,0.0,5.32,0.0,2.44,0.0,5.47,3.91,4.42,1.1


col,All:abs_alpha_hl,All:mean_abs_alpha,All:n_grs_p05,All:n_t196
row,,,,
A:CAPM,0.80,0.26,6.0,12.0
A:CH3,1.23,0.53,13.0,11.0
A:CH4,1.26,0.53,17.0,17.0
A:Car4,0.80,0.28,13.0,19.0
A:FF3,1.02,0.33,14.0,19.0
A:FF5,0.67,0.26,5.0,14.0
A:q,0.65,0.23,4.0,9.0


full


col,Momentum:SHT,Momentum:MHT,Value-versus-growth:SHT,Value-versus-growth:MHT,Investment:SHT,Investment:MHT,Profitability:SHT,Profitability:MHT,Intangibles:SHT,Intangibles:MHT,Trading frictions:SHT,Trading frictions:MHT,All categories:SHT,All categories:MHT
value,53.33,24.44,1.47,0.0,5.56,0.0,42.55,22.34,24.39,3.66,17.19,1.56,24.06,8.17


col,All:abs_alpha_hl,All:mean_abs_alpha,All:n_grs_p05,All:n_t196
row,,,,
A:CAPM,0.61,0.19,61.0,104.0
A:CH3,0.53,0.21,48.0,75.0
A:CH4,0.49,0.20,36.0,69.0
A:Car4,0.65,0.23,91.0,95.0
A:FF3,0.75,0.24,94.0,95.0
A:FF5,0.59,0.20,93.0,94.0
A:q,0.30,0.28,103.0,33.0


## Month-by-month checks against the authors' series (lighthouse 3.3)

In [17]:
fc = pd.read_csv(PROJECT_ROOT / 'results' / 'li_factor_check.csv'); display(fc.round(3))
gc = pd.read_csv(PROJECT_ROOT / 'results' / 'li_g5g1_check.csv')
print(gc['status'].value_counts().to_dict())
gc[gc['status'] == 'miss'].sort_values('corr').round(3)

,factor,months,corr,mean_ours_pct,mean_authors_pct,pass
0,mkt,228,1.000,0.618,0.621,True
1,smbff3,228,0.997,0.644,0.635,True
2,hmlff3,228,0.958,0.318,0.380,True
3,smbff5,228,0.997,0.678,0.676,True
4,hml,228,0.961,0.324,0.419,True
5,rmw,228,0.906,-0.264,-0.142,False
6,cma,228,0.992,0.077,0.055,True
7,me,228,0.996,0.775,0.781,True
8,ia,228,0.965,0.264,0.290,True
9,roe,228,0.980,0.490,0.543,True


{'pass': 99, 'not built': 22, 'miss': 18}


,period,author_name,anomaly,authors_mean_pct,months,corr,ours_mean_pct,corr_g1,corr_g5,status
91,2008_2019,cei,Cei,-0.677,132.0,0.480,-0.898,0.940,0.962,miss
118,2008_2019,psbeta_6,βPS6,-0.442,132.0,0.659,-0.161,0.983,0.993,miss
117,2008_2019,psbeta_3,βPS3,-0.486,132.0,0.670,-0.171,0.983,0.992,miss
30,2000_2019,isq_3,Isq3,-0.251,227.0,0.800,-0.096,0.996,0.996,miss
45,2000_2019,noa,Noa,-0.507,228.0,0.816,-0.579,0.988,0.989,miss
0,2000_2019,_6_res_mom_3,ϵ63,0.556,213.0,0.836,0.610,0.989,0.980,miss
110,2008_2019,i_a,I/A,-0.583,132.0,0.851,-0.213,0.981,0.998,miss
7,2000_2019,ato_q_3,Atoq3,0.633,206.0,0.856,0.613,0.983,0.993,miss
62,2000_2019,rn_2_5_1,"R[2,5]n",-0.602,228.0,0.867,-0.564,0.984,0.987,miss
29,2000_2019,isq_1,Isq1,-0.402,227.0,0.869,-0.314,0.993,0.995,miss
